## imports

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import sklearn

sys.path.append('../src')
from load_clean_data import load_clean_data
from helpers import evaluate_forecast

In [9]:
df = load_clean_data()
df = df.sort_values('Date')
print(df.shape)

c:\Users\corbi\Documents\Projects\sales-forecasting\sales-forecasting-project\notebooks\../src\load_clean_data.py:6: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('../data/train.csv')


(1017209, 18)


### Split Dataset

In [10]:
train = df[df['Date'] < '2015-06-30']
test  = df[df['Date'] >= '2015-06-30']

length_check = len(train) + len(test) == len(df)

print(f"Length check: {length_check}")

Length check: True


#### Store 1

In [11]:
store_1_train = train[train['Store'] == 1]
store_1_test  = test[test['Store'] == 1]

print(f"Store 1 train shape: {store_1_train.shape}")
print(f"Store 1 test shape: {store_1_test.shape}")

Store 1 train shape: (910, 18)
Store 1 test shape: (32, 18)


### Seasonal Average

#### Last 28-Day Average

In [19]:
store_1_28_day_train = store_1_train[store_1_train['Date'] >= '2015-06-02']

print(f"Store 1 train shape (last 28 days): {store_1_28_day_train.shape}")
print(f"Store 1 train shape (last 28 days): {store_1_28_day_train['Date'].min()} to {store_1_28_day_train['Date'].max()}")

Store 1 train shape (last 28 days): (28, 18)
Store 1 train shape (last 28 days): 2015-06-02 00:00:00 to 2015-06-29 00:00:00


In [24]:
store_1_28_day_train.groupby(['Store', 'DayOfWeek']).size()

Store  DayOfWeek
1      1            4
       2            4
       3            4
       4            4
       5            4
       6            4
       7            4
dtype: int64

In [ ]:
seasonal_average_28_day_forecast = store_1_28_day_train.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(seasonal_average_28_day_forecast)

seasonal_average_28_day_forecast = seasonal_average_28_day_forecast.reset_index()

7
Store  DayOfWeek
1      1            4658.00
       2            4541.50
       3            4186.50
       4            2951.25
       5            4149.50
       6            4138.75
       7               0.00
Name: Sales, dtype: float64


In [25]:
seasonal_average_28_day_forecast = seasonal_average_28_day_forecast.rename(columns={'Sales': '28DaySeasonalAverage'})

store_1_test = store_1_test.merge(seasonal_average_28_day_forecast, on=['Store', 'DayOfWeek'], how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['DayOfWeek', 'Date', 'Sales', '28DaySeasonalAverage']].head(10))

(32, 19)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
28DaySeasonalAverage         0
dtype: int64

   DayOfWeek       Date  Sales  28DaySeasonalAverage
0          2 2015-06-30   5735               4541.50
1          3 2015-07-01   5223               4186.50
2          4 2015-07-02   5558               2951.25
3          5 2015-07-03   4665               4149.50
4          6 2015-07-04   4797               4138.75
5          7 2015-07-05      0                  0.00
6          1 201

#### eval

In [ ]:
evaluate_forecast(store_1_test['Sales'], store_1_test['28DaySeasonalAverage'])

RMSE: 910.90
MAE: 704.87
WAPE: 17.76%


#### Full History Average

In [18]:
seasonal_average_full_forecast = store_1_train.groupby(['Store', 'DayOfWeek'])['Sales'].mean()
print(seasonal_average_full_forecast)

seasonal_average_full_forecast = seasonal_average_full_forecast.reset_index()

Store  DayOfWeek
1      1            4945.153846
       2            4652.907692
       3            4456.315385
       4            4077.700000
       5            4518.630769
       6            4963.546154
       7               0.000000
Name: Sales, dtype: float64


In [ ]:
seasonal_average_full_forecast = seasonal_average_full_forecast.rename(columns={'Sales': 'FullSeasonalAverage'})

store_1_test = store_1_test.merge(seasonal_average_full_forecast, on=['Store', 'DayOfWeek'], how='left')

print(store_1_test.shape)
print()
print(store_1_test.isnull().sum())
print()
print(store_1_test[['DayOfWeek', 'Date', 'Sales', 'FullSeasonalAverage']].head(10))

(32, 19)

Store                        0
DayOfWeek                    0
Date                         0
Sales                        0
Customers                    0
Open                         0
Promo                        0
StateHoliday                 0
SchoolHoliday                0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
SeasonalAverage              0
dtype: int64

   DayOfWeek       Date  Sales  SeasonalAverage
0          2 2015-06-30   5735      4652.907692
1          3 2015-07-01   5223      4456.315385
2          4 2015-07-02   5558      4077.700000
3          5 2015-07-03   4665      4518.630769
4          6 2015-07-04   4797      4963.546154
5          7 2015-07-05      0         0.000000
6          1 2015-07-06   4359      4945.153846
7  

#### eval

In [ ]:
evaluate_forecast(store_1_test['Sales'], store_1_test['FullSeasonalAverage'])

RMSE: 702.58
MAE: 567.18
WAPE: 14.29%
